In [1]:
import pandas as pd

# Load the data
df = pd.read_csv('group_means_primary_v2_fs.csv')

print(df.columns)


Index(['A.Age', 'A.Gender', 'A.Education Level', 'A.Monthly Income',
       'A.Marital Status', 'A.Pregnancy Status', 'A.Breastfeeding Status',
       'A.Direct Patient Contact', 'A.Job Type', 'A.Years of Employment',
       'A.COVID-19 Patient Care', 'A.Existing Illnesses', 'A.COVID-19 History',
       'A.Severe was your infection', 'A.Additional Vaccines',
       'A.Immunity Boosters', 'A.COVID-19 Risk Perception',
       'A.COVID-19 Concern Level', 'B.Vaccination Status', 'B.Vaccine Choice',
       'B.Vaccine Side Effects', 'B.Family/Friend Side Effects',
       'B.View of vaccine', 'A.Blog', 'A.Internet', 'A.KK', 'A.LamanMS',
       'A.MediaMassa', 'A.Orang', 'mean_per_efficacy', 'mean_per_trust',
       'C.Non_hesitant', 'B.State', 'A.Ethnicity', 'A.Religion'],
      dtype='object')


In [2]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Label Encode the "State" column
state_encoder = LabelEncoder()
df.loc[:, 'B.State'] = state_encoder.fit_transform(df['B.State'])

# Print the label-encoded "State" feature
print(df['B.State'].head())

0    13
1     8
2    12
3    12
4     9
Name: B.State, dtype: object


In [3]:
# Convert integer to string: survived
df['C.Non_hesitant'] = df['C.Non_hesitant'].astype(str)
df['C.Non_hesitant'].describe()

count     554
unique      2
top         1
freq      321
Name: C.Non_hesitant, dtype: object

In [4]:
df['C.Non_hesitant'].value_counts()

C.Non_hesitant
1    321
0    233
Name: count, dtype: int64

In [5]:
target=df["C.Non_hesitant"]
features=df.drop('C.Non_hesitant',axis=1)

In [6]:
# Split data into train and test sets

# Import train_test_split function
from sklearn.model_selection import train_test_split

# Split the dataset into training + development set and test set
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size = 0.2, random_state = 0)

In [7]:
# Columns for ANOVA test
anova_columns = [
    'A.Age', 'A.Years of Employment', 'mean_per_efficacy', 'mean_per_trust'
]

# Columns for Chi-Squared test
chi2_columns = [
    'A.Gender', 'A.Education Level', 'A.Monthly Income', 'A.Marital Status',
    'A.Pregnancy Status', 'A.Breastfeeding Status', 'A.Direct Patient Contact', 'A.Job Type',
    'A.COVID-19 Patient Care', 'A.Existing Illnesses', 'A.COVID-19 History', 'A.Severe was your infection',
    'A.Additional Vaccines', 'A.Immunity Boosters', 'A.COVID-19 Risk Perception', 'A.COVID-19 Concern Level',
    'B.Vaccination Status', 'B.Vaccine Choice', 'B.Vaccine Side Effects', 'B.Family/Friend Side Effects',
    'B.View of vaccine', 'A.Blog', 'A.Internet', 'A.KK', 'A.LamanMS', 'A.MediaMassa',
    'A.Orang', 'B.State', 'A.Ethnicity', 'A.Religion'
]

# Extract and save ANOVA columns from the training set
anova_train = x_train[anova_columns]

# Extract and save Chi-Squared columns from the training set
chi2_train = x_train[chi2_columns]


# ANOVA TESTING

In [8]:
# Import SelectKBest and chi2 modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif


# Apply SelectKBest with ANOVA on the anova_train data
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(anova_train, y_train)
scores = selector.scores_
p_values = selector.pvalues_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': anova_train.columns,
    'ANOVA Score': scores,
    'P-value': p_values
})

# Sort the DataFrame by ANOVA scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='ANOVA Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                 Feature  ANOVA Score       P-value
2      mean_per_efficacy   365.654712  8.398177e-60
3         mean_per_trust    66.674522  3.397527e-15
1  A.Years of Employment     0.413693  5.204351e-01
0                  A.Age     0.003264  9.544689e-01


In [9]:
# Select features with p-value < 0.05
significant_features_anova = feature_scores[feature_scores['P-value'] < 0.05]

# Display significant features
print(significant_features_anova)

             Feature  ANOVA Score       P-value
2  mean_per_efficacy   365.654712  8.398177e-60
3     mean_per_trust    66.674522  3.397527e-15


# CHI2 TESTING

In [10]:
# Currently the target is integer data type
y_train.dtype

dtype('O')

In [11]:
# Import SelectKBest and chi2 modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2


# Apply SelectKBest with chi2
selector = SelectKBest(score_func=chi2, k='all')
selector.fit(chi2_train, y_train)
scores = selector.scores_
p_values = selector.pvalues_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': chi2_train.columns,
    'Chi-squared Score': scores,
    'P-value': p_values
})

# Sort the DataFrame by Chi-squared scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='Chi-squared Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                         Feature  Chi-squared Score       P-value
20             B.View of vaccine          78.102735  9.781842e-19
2               A.Monthly Income          16.497914  4.870362e-05
25                  A.MediaMassa          11.994572  5.335572e-04
19  B.Family/Friend Side Effects          10.968880  9.265475e-04
28                   A.Ethnicity           5.623181  1.772445e-02
21                        A.Blog           4.459069  3.471642e-02
13           A.Immunity Boosters           4.176471  4.098897e-02
27                       B.State           3.725856  5.357614e-02
8        A.COVID-19 Patient Care           3.595842  5.792428e-02
16          B.Vaccination Status           2.797565  9.440759e-02
24                     A.LamanMS           2.632523  1.046952e-01
12         A.Additional Vaccines           2.089297  1.483343e-01
29                    A.Religion           1.940813  1.635803e-01
14    A.COVID-19 Risk Perception           1.710826  1.908783e-01
17        

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [12]:
# Select features with p-value < 0.05
significant_features_chi = feature_scores[feature_scores['P-value'] < 0.05]

# Display significant features
print(significant_features_chi)

                         Feature  Chi-squared Score       P-value
2               A.Monthly Income          16.497914  4.870362e-05
13           A.Immunity Boosters           4.176471  4.098897e-02
19  B.Family/Friend Side Effects          10.968880  9.265475e-04
20             B.View of vaccine          78.102735  9.781842e-19
21                        A.Blog           4.459069  3.471642e-02
25                  A.MediaMassa          11.994572  5.335572e-04
28                   A.Ethnicity           5.623181  1.772445e-02


# INFORMATION GAIN 

In [13]:
# Import SelectKBest and information gain modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

# Apply SelectKBest with information gain
selector = SelectKBest(score_func=mutual_info_classif, k='all')
selector.fit(chi2_train, y_train)
scores = selector.scores_ 

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': chi2_train.columns,
    'Information gain Score': scores,
})

# Sort the DataFrame by Chi-squared scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='Information gain Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                         Feature  Information gain Score
20             B.View of vaccine                0.075272
14    A.COVID-19 Risk Perception                0.041749
26                       A.Orang                0.040834
23                          A.KK                0.035043
17              B.Vaccine Choice                0.033721
16          B.Vaccination Status                0.032251
1              A.Education Level                0.032215
12         A.Additional Vaccines                0.029388
10            A.COVID-19 History                0.027896
19  B.Family/Friend Side Effects                0.027874
0                       A.Gender                0.026120
4             A.Pregnancy Status                0.022256
27                       B.State                0.019950
2               A.Monthly Income                0.018577
15      A.COVID-19 Concern Level                0.014252
25                  A.MediaMassa                0.009030
3               A.Marital Statu

In [14]:
selected_features_in = feature_scores_sorted[feature_scores_sorted['Information gain Score'] > 0.05]
print(selected_features_in)

              Feature  Information gain Score
20  B.View of vaccine                0.075272


# OUTPUT

In [15]:
result1 = pd.concat([significant_features_anova,significant_features_chi])
result1 = result1.drop(columns=['ANOVA Score', 'P-value', 'Chi-squared Score'])
print (result1)

                         Feature
2              mean_per_efficacy
3                 mean_per_trust
2               A.Monthly Income
13           A.Immunity Boosters
19  B.Family/Friend Side Effects
20             B.View of vaccine
21                        A.Blog
25                  A.MediaMassa
28                   A.Ethnicity


In [16]:
significant_features = result1['Feature'].tolist()
anova_chi = df[significant_features + ['C.Non_hesitant']]
anova_chi.to_csv('anova_chi.csv', index=False)

In [17]:
result2 = pd.concat([significant_features_anova,selected_features_in])
result2 = result2.drop(columns=['ANOVA Score', 'P-value', 'Information gain Score'])
print (result2)

              Feature
2   mean_per_efficacy
3      mean_per_trust
20  B.View of vaccine


In [18]:
significant_features = result2['Feature'].tolist()
anova_in = df[significant_features + ['C.Non_hesitant']]
anova_in.to_csv('anova_IG.csv', index=False)